<a href="https://colab.research.google.com/github/uguazelli/scikit_learn/blob/main/XGBoost101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

XGBoost: A Powerful Boosting Algorithm
XGBoost (eXtreme Gradient Boosting) is an optimized distributed gradient boosting library designed to be highly efficient, flexible, and portable. It implements machine learning algorithms under the Gradient Boosting framework.

🧠 The Math Behind XGBoost (Simplified)
Imagine you're trying to predict something, like whether a customer will buy a product.

Start Simple: You begin with a very basic guess. Let's say your initial guess for everyone is "they won't buy." This guess will be wrong for many people.

Find the Errors: For each person, you calculate how wrong your initial guess was. These "wrongs" or "residuals" are the errors your first simple guess made.

Learn from Errors: Instead of trying to predict the original outcome (buy/not buy) directly again, you now train a new, simple model (often a small decision tree) to predict these errors. This new model tries to figure out patterns in why your first guess was wrong.

Correct and Combine: You then take the predictions from this "error-correcting" model and add them to your initial guess. This gives you a slightly better, combined prediction.

Repeat and Refine: You repeat steps 2-4 many, many times. Each new simple model learns from the remaining errors of the combined predictions from all previous models. It's like having a team of experts, where each new expert focuses on fixing the mistakes of the previous experts.

"Gradient" Part: The "gradient" in Gradient Boosting refers to a fancy way of finding the direction of the "errors" to learn from. It's like finding the steepest path down a hill to minimize the mistakes.

"Boosting" Part: This refers to the idea of sequentially building models, where each new model boosts the performance of the overall system by focusing on the previous models' weaknesses.

"eXtreme" Part: XGBoost takes this concept to the "eXtreme" by adding:

Regularization: It prevents individual simple models from becoming too complex and overfitting the data (like telling each expert not to memorize every single mistake, but to learn general patterns).

Parallel Processing: It's designed to run very fast by using your computer's resources efficiently.

Handling Missing Values: It has built-in ways to deal with incomplete data.

Tree Pruning: It intelligently cuts back parts of the simple decision trees that aren't contributing much.

In essence, XGBoost builds a strong, complex model by combining many simple, weak models, each correcting the mistakes of its predecessors.

💻 XGBoost Example in Python (Colab Friendly)
This example will demonstrate how to use XGBoost for a binary classification problem. We'll generate some synthetic data, train an XGBClassifier, evaluate its performance, and then show how to use the trained model for new predictions.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.datasets import load_breast_cancer # To load a real dataset

In [ ]:
# --- 1. Load a Real Dataset (Breast Cancer Wisconsin) ---
# This dataset is commonly used for binary classification.
# It contains features computed from a digitized image of a fine needle aspirate (FNA)
# of a breast mass. The target variable indicates whether the mass is malignant (0) or benign (1).
print("1. Loading the Breast Cancer Wisconsin dataset...")
cancer = load_breast_cancer()

# Convert to Pandas DataFrame for easier handling
X_df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y_series = pd.Series(cancer.target, name='target')

print(f"Dataset loaded: {X_df.shape[0]} samples, {X_df.shape[1]} features.")
print("Target classes: (0: malignant, 1: benign)")
print("\nFirst 5 rows of features:")
print(X_df.head())
print("\nFirst 5 rows of target:")
print(y_series.head())
print(f"\nTarget distribution:\n{y_series.value_counts()}")

1. Loading the Breast Cancer Wisconsin dataset...
Dataset loaded: 569 samples, 30 features.
Target classes: (0: malignant, 1: benign)

First 5 rows of features:
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4      

In [ ]:
# --- 2. Split Data into Training and Testing Sets ---
# We'll use 80% for training and 20% for testing.
# stratify=y ensures that the proportion of classes is the same in both train and test sets,
# which is important for imbalanced datasets.
print("\n2. Splitting data into training and testing sets...")
X_train, X_test, y_train, y_test = train_test_split(
    X_df, y_series, test_size=0.2, random_state=42, stratify=y_series
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")
print(f"Training target distribution:\n{y_train.value_counts(normalize=True)}")
print(f"Testing target distribution:\n{y_test.value_counts(normalize=True)}")


2. Splitting data into training and testing sets...
Training set size: 455 samples
Testing set size: 114 samples
Training target distribution:
target
1    0.626374
0    0.373626
Name: proportion, dtype: float64
Testing target distribution:
target
1    0.631579
0    0.368421
Name: proportion, dtype: float64


In [ ]:
# --- 3. Initialize and Train the XGBoost Classifier ---
# We'll use XGBClassifier for binary classification.
# Common parameters:
#   - objective: 'binary:logistic' for binary classification (outputs probabilities)
#   - use_label_encoder: Set to False to suppress a future deprecation warning
#   - eval_metric: Metric to monitor during training (e.g., 'logloss' for classification)
#   - n_estimators: Number of boosting rounds (trees)
#   - learning_rate: Step size shrinkage to prevent overfitting
#   - max_depth: Maximum depth of a tree
print("\n3. Initializing and training the XGBoost Classifier...")
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    use_label_encoder=False, # Suppress warning
    eval_metric='logloss',   # Evaluation metric
    n_estimators=100,        # Number of boosting rounds (trees)
    learning_rate=0.1,       # Step size shrinkage
    max_depth=3,             # Maximum depth of a tree
    random_state=42          # For reproducibility
)

# Train the model
# You can also use early stopping to prevent overfitting if you have a validation set
# xgb_model.fit(X_train, y_train, early_stopping_rounds=10, eval_set=[(X_test, y_test)], verbose=False)
xgb_model.fit(X_train, y_train)

print("XGBoost model training complete.")


3. Initializing and training the XGBoost Classifier...


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [20:53:25] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBoost model training complete.


In [ ]:
# --- 4. Make Predictions and Evaluate the Model ---
print("\n4. Making predictions and evaluating the model...")

# Predict probabilities (useful for thresholding or ROC curves)
# predict_proba returns probabilities for each class. We take the probability of the positive class (1).
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

# Predict class labels (0 or 1)
y_pred = xgb_model.predict(X_test)

# Calculate Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy on the test set: {accuracy:.4f}")

# Display Classification Report (precision, recall, f1-score)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

# Display Confusion Matrix
print("\nConfusion Matrix:")
# Rows are actual, columns are predicted
# [[True Negative, False Positive],
#  [False Negative, True Positive]]
print(confusion_matrix(y_test, y_pred))


4. Making predictions and evaluating the model...
Accuracy on the test set: 0.9474

Classification Report:
              precision    recall  f1-score   support

   malignant       0.95      0.90      0.93        42
      benign       0.95      0.97      0.96        72

    accuracy                           0.95       114
   macro avg       0.95      0.94      0.94       114
weighted avg       0.95      0.95      0.95       114


Confusion Matrix:
[[38  4]
 [ 2 70]]


In [ ]:
# --- 5. Example of Usage After Training (Predicting on New Data) ---
print("\n5. Example of usage after training: Predicting on new, unseen data.")

# Create some hypothetical new data points.
# IMPORTANT: The new data MUST have the same features (columns) and order
# as the training data. Here we take the first two samples from the original dataset
# and pretend they are new, unseen data.
new_data_samples = X_df.iloc[[0, 10, 20]] # Select a few existing samples as "new data"
new_data = pd.DataFrame(new_data_samples, columns=cancer.feature_names)

print("\nNew data to predict on (first 3 samples from original dataset):")
print(new_data)


5. Example of usage after training: Predicting on new, unseen data.

New data to predict on (first 3 samples from original dataset):
    mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0         17.99         10.38          122.80     1001.0          0.11840   
10        16.02         23.24          102.70      797.8          0.08206   
20        13.08         15.71           85.63      520.0          0.10750   

    mean compactness  mean concavity  mean concave points  mean symmetry  \
0            0.27760         0.30010              0.14710         0.2419   
10           0.06669         0.03299              0.03323         0.1528   
20           0.12700         0.04568              0.03110         0.1967   

    mean fractal dimension  ...  worst radius  worst texture  worst perimeter  \
0                  0.07871  ...         25.38          17.33           184.60   
10                 0.05697  ...         19.19          33.88           123.80   
20       

In [ ]:
# Make predictions on the new data
new_predictions = xgb_model.predict(new_data)
new_probabilities = xgb_model.predict_proba(new_data)[:, 1] # Probability of being class 1 (benign)

print("\nPredictions for new data:")
for i, pred_class_idx in enumerate(new_predictions):
    predicted_class_name = cancer.target_names[pred_class_idx]
    actual_class_name = cancer.target_names[y_series.iloc[new_data_samples.index[i]]]
    print(f"Sample {new_data_samples.index[i]} (Actual: {actual_class_name}): "
          f"Predicted Class = {predicted_class_name}, "
          f"Probability of Benign (Class 1) = {new_probabilities[i]:.4f}")

# You can interpret these predictions:
# If Predicted Class = 'malignant', the model thinks it's malignant.
# If Predicted Class = 'benign', the model thinks it's benign.
# The Probability of Benign (Class 1) tells you how confident the model is about it being benign.
# (e.g., a probability > 0.5 typically means 'benign', < 0.5 means 'malignant')


Predictions for new data:
Sample 0 (Actual: malignant): Predicted Class = malignant, Probability of Benign (Class 1) = 0.0249
Sample 10 (Actual: malignant): Predicted Class = malignant, Probability of Benign (Class 1) = 0.0444
Sample 20 (Actual: benign): Predicted Class = benign, Probability of Benign (Class 1) = 0.9988


✅ Summary
XGBoost is a powerful and popular machine learning algorithm, especially effective for structured data (like tables). It works by iteratively building a series of simple models, each one correcting the errors of the previous ones. Its "eXtreme" optimizations make it fast and robust, while its ability to learn from errors makes it highly accurate. Once trained, you can easily use the model to make predictions on new, unseen data, providing classifications or regression values based on your problem.